# 03 · Preparación: diccionario de etiquetas v1, deduplicación y splits — Grupo 17
**Paso 4 del Sprint 2.** Entrada: `data/interim/emails_v0.parquet` (EDA). Salidas:
`data/processed/emails_v1.parquet`, `data/meta/diccionario_etiquetas_v1.json` y el `RESUMEN`.

Cada exclusión queda registrada con su motivo y su evidencia del EDA. Las decisiones están en la Celda 2 (`CONFIG`)
para poder revertirlas cambiando una línea, sin tocar código.

**Criterios de cierre:** (1) toda fila excluida tiene motivo, (2) 0 hashes compartidos entre train/val/test,
(3) cada clase con ≥ 2 orígenes, (4) 0 IOCs vivos en `texto_modelo`, (5) folds por fuente y temporal definidos.

In [1]:
# Celda 1: Entorno y módulos
import os, sys, json, random, importlib.util
from pathlib import Path
import numpy as np, pandas as pd
SEED = 17; random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
elif ENV == "kaggle":
    BASE = Path("/kaggle/working/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
PROC, META = BASE / "data/processed", BASE / "data/meta"
PROC.mkdir(parents=True, exist_ok=True)
def cargar(nombre, ruta):
    spec = importlib.util.spec_from_file_location(nombre, ruta); m = importlib.util.module_from_spec(spec)
    sys.modules[nombre] = m; spec.loader.exec_module(m); return m
ioc = cargar("ioc_utils", BASE / "src/ioc_utils.py"); dp = cargar("data_prep", BASE / "src/data_prep.py")
assert "dominio_propio" in dp.MARCADORES, "Falta data_prep v0.5 en src/"
E = pd.read_parquet(BASE / "data/interim/emails_v0.parquet")
print(ENV, "| filas de entrada:", len(E))

Mounted at /content/drive
colab | filas de entrada: 94566


In [2]:
# Celda 2: Decisiones (con su evidencia del EDA)
CONFIG = {
  "grupos_excluidos": {
    "Nazario": "Q2+Q5: mismos años (2015–2022) que Nazario anual; 439 hashes idénticos y el resto difiere por formateo. "
               "Q4: la versión CSV perdió los enlaces (16,2 % con URL vs 92,1 % en el crudo). Se usa el crudo.",
    "Enron":   "Q4: 18,8 % contiene 'http' pero 0,0 % de URLs válidas: texto tokenizado que destruye las URLs; "
               "Q5: sin fecha. Formato distinto al resto: riesgo de que el modelo aprenda el formato del corpus.",
    "Ling":    "Q4: 38,8 % contiene 'http' pero 0,0 % de URLs válidas (mismo problema que Enron); Q5: sin fecha.",
  },
  "grupos_con_filtro_idioma": ["phishing_pot", "nazario_anual"],   # Q3: pot 51,1 % inglés; idioma medido en el 100 %
  "idioma": "en",
  "min_len_plano": 20,                                            # Q2: 39 textos casi vacíos
  "clases": {"legitimo": 0, "spam": 1, "phishing": 2},
  "proporciones_split": {"train": 0.70, "val": 0.15, "test": 0.15},
  "anio_corte_temporal": 2025,
  # sonda de fuga (03, 1.ª corrida limpia): "monkey" en el 23 % de Nazario anual = dominio del buzón del autor
  "dominios_propios": {"nazario_anual": ["monkey.org", "monkey"]},                                    # fold temporal: phishing 2025+ como no visto
}
CONFIG

{'grupos_excluidos': {'Nazario': 'Q2+Q5: mismos años (2015–2022) que Nazario anual; 439 hashes idénticos y el resto difiere por formateo. Q4: la versión CSV perdió los enlaces (16,2 % con URL vs 92,1 % en el crudo). Se usa el crudo.',
  'Enron': "Q4: 18,8 % contiene 'http' pero 0,0 % de URLs válidas: texto tokenizado que destruye las URLs; Q5: sin fecha. Formato distinto al resto: riesgo de que el modelo aprenda el formato del corpus.",
  'Ling': "Q4: 38,8 % contiene 'http' pero 0,0 % de URLs válidas (mismo problema que Enron); Q5: sin fecha."},
 'grupos_con_filtro_idioma': ['phishing_pot', 'nazario_anual'],
 'idioma': 'en',
 'min_len_plano': 20,
 'clases': {'legitimo': 0, 'spam': 1, 'phishing': 2},
 'proporciones_split': {'train': 0.7, 'val': 0.15, 'test': 0.15},
 'anio_corte_temporal': 2025,
 'dominios_propios': {'nazario_anual': ['monkey.org', 'monkey']}}

In [3]:
# Celda 3: Exclusiones en orden, con motivo por fila
E = E.sort_values(["grupo", "fuente", "hash"], kind="stable").reset_index(drop=True)  # orden determinista
E["motivo_exclusion"] = None
def excluir(mascara, motivo):
    m = mascara & E["motivo_exclusion"].isna(); E.loc[m, "motivo_exclusion"] = motivo; return int(m.sum())
bitacora = {}
bitacora["grupo_excluido"] = excluir(E["grupo"].isin(CONFIG["grupos_excluidos"]), "grupo_excluido")
bitacora["casi_vacio"]     = excluir(E["len_plano"] < CONFIG["min_len_plano"], "casi_vacio")
bitacora["idioma"]         = excluir(E["grupo"].isin(CONFIG["grupos_con_filtro_idioma"]) & (E["idioma"] != CONFIG["idioma"]), "idioma")
vivos = E["motivo_exclusion"].isna()
# duplicado exacto (mismo hash) entre las filas que siguen vivas; se conserva la primera en el orden determinista
dup = vivos & E[vivos].duplicated("hash", keep="first").reindex(E.index, fill_value=False)
bitacora["duplicado"]      = excluir(dup, "duplicado")
exclusiones_por_grupo = pd.crosstab(E["grupo"], E["motivo_exclusion"].fillna("CONSERVADO"))
display(exclusiones_por_grupo); bitacora

motivo_exclusion,CONSERVADO,casi_vacio,duplicado,grupo_excluido,idioma
grupo,,,,,
CEAS_08,38963,0,191,0,0
Enron,0,0,0,29767,0
Ling,0,0,0,2859,0
Nazario,0,0,0,1565,0
Nigerian_Fraud,3290,1,41,0,0
SpamAssasin,5802,0,7,0,0
nazario_anual,2927,8,330,0,201
phishing_pot,3702,21,696,0,4195


{'grupo_excluido': 34191, 'casi_vacio': 30, 'idioma': 4396, 'duplicado': 1265}

In [4]:
# Celda 4: Dataset v1: clase numérica y texto de entrada del modelo
D = E[E["motivo_exclusion"].isna()].copy()
D["y"] = D["clase"].map(CONFIG["clases"]).astype(int)
D["texto_modelo"] = [dp.texto_para_modelo(t, CONFIG["dominios_propios"].get(g, ())) for t, g in zip(D["texto_defang"], D["grupo"])]
assert not D["texto_modelo"].map(ioc.iocs_vivos).map(bool).any(), "IOC vivo en texto_modelo"
D["id"] = D["hash"].str[:16]
assert D["id"].is_unique
tabla_final = pd.crosstab(D["grupo"], D["clase"], margins=True)
origenes_por_clase = D.groupby("clase")["grupo"].nunique().to_dict()
assert min(origenes_por_clase.values()) >= 2, f"Una clase quedó con un solo origen: {origenes_por_clase}"
display(tabla_final); origenes_por_clase

clase,legitimo,phishing,spam,All
grupo,,,,
CEAS_08,17289,0,21674,38963
Nigerian_Fraud,0,3290,0,3290
SpamAssasin,4090,0,1712,5802
nazario_anual,0,2927,0,2927
phishing_pot,0,3702,0,3702
All,21379,9919,23386,54684


{'legitimo': 2, 'phishing': 3, 'spam': 2}

In [5]:
# Celda 5: Split estratificado por clase×grupo y folds de evaluación
from sklearn.model_selection import train_test_split
estrato = D["clase"] + "|" + D["grupo"]
p = CONFIG["proporciones_split"]
idx_tr, idx_tmp = train_test_split(D.index, test_size=1 - p["train"], stratify=estrato, random_state=SEED)
idx_va, idx_te = train_test_split(idx_tmp, test_size=p["test"] / (p["val"] + p["test"]),
                                  stratify=estrato.loc[idx_tmp], random_state=SEED)
D["split"] = "train"; D.loc[idx_va, "split"] = "val"; D.loc[idx_te, "split"] = "test"
assert D.groupby("hash")["split"].nunique().max() == 1      # criterio 2 (garantizado por la deduplicación)

# Regla genérica de fold "fuera de distribución": lo que cae en la máscara va COMPLETO a test y sale de train/val;
# el resto conserva su split. Así el test mezcla clases y además se reporta la métrica solo sobre lo "fuera".
def fold(mascara_fuera):
    s = D["split"].copy(); s[mascara_fuera] = "test"; return s
FOLDS = {f"loso_{g}": D["grupo"] == g for g in sorted(D["grupo"].unique())}
FOLDS["temporal_phishing_2025+"] = (D["clase"] == "phishing") & (D["anio"] >= CONFIG["anio_corte_temporal"])
for nombre, m in FOLDS.items():
    D[f"fold_{nombre}"] = fold(m)
    D[f"fuera_{nombre}"] = m
tam_folds = {n: {"fuera": int(m.sum()), "train_restante": int(((D[f"fold_{n}"] == "train")).sum()),
                 "clases_fuera": D.loc[m, "clase"].value_counts().to_dict()} for n, m in FOLDS.items()}
tabla_split = pd.crosstab(D["split"], D["clase"])
display(tabla_split); tam_folds

clase,legitimo,phishing,spam
split,,,
test,3207,1488,3508
train,14965,6943,16370
val,3207,1488,3508


{'loso_CEAS_08': {'fuera': 38963,
  'train_restante': 11004,
  'clases_fuera': {'spam': 21674, 'legitimo': 17289}},
 'loso_Nigerian_Fraud': {'fuera': 3290,
  'train_restante': 35975,
  'clases_fuera': {'phishing': 3290}},
 'loso_SpamAssasin': {'fuera': 5802,
  'train_restante': 34217,
  'clases_fuera': {'legitimo': 4090, 'spam': 1712}},
 'loso_nazario_anual': {'fuera': 2927,
  'train_restante': 36229,
  'clases_fuera': {'phishing': 2927}},
 'loso_phishing_pot': {'fuera': 3702,
  'train_restante': 35687,
  'clases_fuera': {'phishing': 3702}},
 'temporal_phishing_2025+': {'fuera': 2040,
  'train_restante': 36856,
  'clases_fuera': {'phishing': 2040}}}

In [6]:
# Celda 6: Guardar, diccionario v1 y RESUMEN
cols = ["id", "fuente", "grupo", "clase", "y", "anio", "hash", "n_urls", "sin_url", "es_html", "len_plano",
        "texto_modelo", "urls_defang", "split"] + [c for c in D.columns if c.startswith(("fold_", "fuera_"))]
D[cols].to_parquet(PROC / "emails_v1.parquet", index=False)
diccionario = {
  "version": "v1", "entrada": "data/interim/emails_v0.parquet", "salida": "data/processed/emails_v1.parquet",
  "mapeo": {"CEAS_08/SpamAssasin label 0": "legitimo", "CEAS_08/SpamAssasin label 1": "spam",
            "Nigerian_Fraud": "phishing (fraude por ingeniería social; métricas se reportan con y sin él)",
            "nazario_anual": "phishing", "phishing_pot": "phishing"},
  "clases": CONFIG["clases"], "decisiones": CONFIG, "bitacora_exclusiones": bitacora,
  "texto_modelo": "data_prep.texto_para_modelo(ioc_utils.defang_texto(asunto+cuerpo)): sin HTML; URL/IP/email/año -> marcadores",
  "rama_url_decision": "Variante B (04/M6): XGBoost sobre URLs de correos con etiqueta heredada del correo. "
                       "PhiUSIIL queda como resultado negativo documentado (M1–M4: artefacto esquema/www/ruta vacía).",
  "rama_url_evidencia_phiusiil": {
    "PhiUSIIL": "solo URL + label invertida; se descartan FILENAME (Q7: prefijo delata phishing en ~45 % de los casos), "
                "URLSimilarityIndex (Q7: =100 en 100 % legítimo vs 0,8 % phishing) y toda feature de página; deduplicar URLs",
    "artefacto_esquema_www": "Q7: legítimo 100 % https y 100 % con www vs phishing 49,2 % y 41,4 %: normalizar esquema y www "
                             "antes de calcular features y validar sobre URLs extraídas de correos"},
}
(META / "diccionario_etiquetas_v1.json").write_text(json.dumps(diccionario, indent=1, ensure_ascii=False, default=str))
import re as _re
sonda_fuga = {g: {t: round(100 * float(sub.str.contains(rf"\b{t}\b", regex=True).mean()), 2) for t in dp.TERMINOS_FUGA}
              for g, sub in D.groupby("grupo")["texto_modelo"].apply(lambda x: x.str.lower()).groupby(level=0)}
sonda_fuga = {g: {t: v for t, v in d.items() if v > 0} for g, d in sonda_fuga.items()}
print("===== RESUMEN (03) =====")
print(json.dumps({"entorno": ENV, "filas_entrada": int(len(E)), "filas_v1": int(len(D)), "bitacora": bitacora,
                  "clase_x_grupo": tabla_final.to_dict(), "origenes_por_clase": origenes_por_clase,
                  "split_x_clase": tabla_split.to_dict(), "folds": tam_folds,
                  "pct_sin_url_por_clase": D.groupby("clase")["sin_url"].mean().mul(100).round(1).to_dict(),
                  "sonda_fuga_pct": sonda_fuga, "marcadores_pct": {k: round(100 * float(D["texto_modelo"].str.contains(v, regex=False).mean()), 1)
                                     for k, v in dp.MARCADORES.items()}},
                 ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (03) =====
{"entorno":"colab","filas_entrada":94566,"filas_v1":54684,"bitacora":{"grupo_excluido":34191,"casi_vacio":30,"idioma":4396,"duplicado":1265},"clase_x_grupo":{"legitimo":{"CEAS_08":17289,"Nigerian_Fraud":0,"SpamAssasin":4090,"nazario_anual":0,"phishing_pot":0,"All":21379},"phishing":{"CEAS_08":0,"Nigerian_Fraud":3290,"SpamAssasin":0,"nazario_anual":2927,"phishing_pot":3702,"All":9919},"spam":{"CEAS_08":21674,"Nigerian_Fraud":0,"SpamAssasin":1712,"nazario_anual":0,"phishing_pot":0,"All":23386},"All":{"CEAS_08":38963,"Nigerian_Fraud":3290,"SpamAssasin":5802,"nazario_anual":2927,"phishing_pot":3702,"All":54684}},"origenes_por_clase":{"legitimo":2,"phishing":3,"spam":2},"split_x_clase":{"legitimo":{"test":3207,"train":14965,"val":3207},"phishing":{"test":1488,"train":6943,"val":1488},"spam":{"test":3508,"train":16370,"val":3508}},"folds":{"loso_CEAS_08":{"fuera":38963,"train_restante":11004,"clases_fuera":{"spam":21674,"legitimo":17289}},"loso_Nigerian_Fraud":{"fuer